# 01. Exploración y ETL (Extract, Transform, Load)
Este notebook contiene el proceso de carga de datos crudos (MINSA, SENAMHI, NASA POWER), limpieza de valores atípicos y nulos, ingeniería de características (lags temporales t-1, t-2) y el etiquetado de la variable objetivo evitando el sobreajuste y la filtración de datos (Data Leakage).

In [1]:
%pip install pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import os
import warnings
import re
warnings.filterwarnings('ignore')

# Definicion de rutas relativas para el entorno de trabajo
ruta_dengue_raw = "../data/raw/dataset_dengue_actualizado.csv"
ruta_clima_dir = "../data/raw/Clima_Piura/"

In [3]:
lista_dataframes = []
print("Iniciando proceso ETL de series climaticas...")

if os.path.exists(ruta_clima_dir):
    for raiz, directorios, archivos in os.walk(ruta_clima_dir):
        for archivo in archivos:
            if archivo.endswith(('.csv', '.CSV')):
                ruta_archivo = os.path.join(raiz, archivo)
                
                # Extraer nombre base del archivo sin extension
                nombre_base = os.path.splitext(archivo)[0].upper()
                
                with open(ruta_archivo, 'r', encoding='utf-8', errors='ignore') as f:
                    primeras_lineas = [next(f, '') for _ in range(15)]
                texto_inicio = "".join(primeras_lineas)
                
                try:
                    # 1. Procesamiento para archivos NASA POWER
                    if "-BEGIN HEADER-" in texto_inicio or "NASA/POWER" in texto_inicio:
                        skip = 0
                        with open(ruta_archivo, 'r', encoding='utf-8', errors='ignore') as f:
                            for i, line in enumerate(f):
                                if "-END HEADER-" in line:
                                    skip = i + 1
                                    break
                        df = pd.read_csv(ruta_archivo, skiprows=skip, sep=r'[,;\t\s]+', engine='python', encoding='utf-8')
                        df.columns = df.columns.str.strip()
                        df['FECHA'] = pd.to_datetime(df['YEAR'].astype(str) + df['DOY'].astype(str).str.zfill(3), format='%Y%j', errors='coerce')
                        df = df.rename(columns={'T2M_MAX': 'MAX', 'T2M_MIN': 'MIN', 'PRECTOTCORR': 'PRECIPITACION'})
                        df = df.replace(-999.0, np.nan)
                        
                    # 2. Procesamiento para archivos SENAMHI NUEVO FORMATO (2023)
                    elif "Año;Mes;Día" in texto_inicio or "Temp. Máx" in texto_inicio:
                        df = pd.read_csv(ruta_archivo, sep=';', encoding='utf-8')
                        df['FECHA'] = pd.to_datetime(
                            df['Año'].astype(str) + '-' + 
                            df['Mes'].astype(str).str.zfill(2) + '-' + 
                            df['Día'].astype(str).str.zfill(2), 
                            errors='coerce'
                        )
                        df = df.rename(columns={
                            df.columns[3]: 'MAX', 
                            df.columns[4]: 'MIN', 
                            df.columns[6]: 'PRECIPITACION'
                        })
                        
                        if df['MAX'].dtype == object:
                            df['MAX'] = pd.to_numeric(df['MAX'].astype(str).str.replace('S/D', 'NaN', regex=False), errors='coerce')
                        if df['MIN'].dtype == object:
                            df['MIN'] = pd.to_numeric(df['MIN'].astype(str).str.replace('S/D', 'NaN', regex=False), errors='coerce')
                            
                        df['PRECIPITACION'] = df['PRECIPITACION'].astype(str).str.replace('T', '0.0', regex=False).str.replace('S/D', 'NaN', regex=False)
                        df['PRECIPITACION'] = pd.to_numeric(df['PRECIPITACION'], errors='coerce')
                        
                    # 3. Procesamiento para archivos SENAMHI ANTIGUO FORMATO
                    else:
                        skip = 0
                        with open(ruta_archivo, 'r', encoding='utf-8', errors='ignore') as f:
                            for i, line in enumerate(f):
                                if "AÑO / MES / DÍA" in line or "AÑO" in line:
                                    skip = i
                                    break
                        linea_ejemplo = primeras_lineas[skip+1] if skip+1 < len(primeras_lineas) else ""
                        separador = ';' if ';' in linea_ejemplo else '\t' if '\t' in linea_ejemplo else ','
                        
                        df = pd.read_csv(ruta_archivo, skiprows=skip+2, header=None, sep=separador, encoding='utf-8')
                        
                        if len(df.columns) >= 5:
                            df = df.iloc[:, [0, 1, 2, 4]]
                            df.columns = ['FECHA', 'MAX', 'MIN', 'PRECIPITACION']
                        elif len(df.columns) == 2:
                            df.columns = ['FECHA', 'PRECIPITACION']
                            df['MAX'] = np.nan
                            df['MIN'] = np.nan
                        
                        df['FECHA'] = pd.to_datetime(df['FECHA'], errors='coerce')
                        
                        if df['MAX'].dtype == object:
                            df['MAX'] = pd.to_numeric(df['MAX'].astype(str).str.replace('S/D', 'NaN', regex=False), errors='coerce')
                        if df['MIN'].dtype == object:
                            df['MIN'] = pd.to_numeric(df['MIN'].astype(str).str.replace('S/D', 'NaN', regex=False), errors='coerce')
                        
                        if 'PRECIPITACION' in df.columns:
                            df['PRECIPITACION'] = df['PRECIPITACION'].astype(str).str.replace('T', '0.0', regex=False).str.replace('S/D', 'NaN', regex=False)
                            df['PRECIPITACION'] = pd.to_numeric(df['PRECIPITACION'], errors='coerce')
                    
                    # 1. Quitar prefijo NASA_ si existe
                    distrito_limpio = nombre_base.replace('NASA_', '')
                    # 2. Quitar el ano y mes del final mediante Regex (ej. _2023_01, _2024, etc)
                    distrito_limpio = re.sub(r'(_\d{4})(_\d{2})?$', '', distrito_limpio)
                    # 3. Quitar sufijo SENAMHI y CSV si quedaron en el nombre
                    distrito_limpio = distrito_limpio.replace('_SENAMHI', '').replace('CSV', '')
                    
                    df['DISTRITO'] = distrito_limpio
                    df = df.dropna(subset=['FECHA'])
                    df = df[['DISTRITO', 'FECHA', 'MAX', 'MIN', 'PRECIPITACION']]
                    lista_dataframes.append(df)
                    
                except Exception as e:
                    print(f"Error de lectura en el archivo {nombre_base}: {e}")
else:
    print(f"Advertencia: La ruta {ruta_clima_dir} no se encuentra disponible. Verifique la ubicacion de los datos.")

Iniciando proceso ETL de series climaticas...


In [4]:
# UNIFICACION Y AGREGACION ESPACIAL
from IPython.display import display

if len(lista_dataframes) == 0:
    print("\nFallo en la extraccion de datos climaticos.")
else:
    df_clima_diario = pd.concat(lista_dataframes, ignore_index=True)
    df_clima_diario = df_clima_diario.sort_values(by=['DISTRITO', 'FECHA'])

    distritos_unicos = df_clima_diario['DISTRITO'].unique()
    print(f"Diagnostico - Numero de distritos procesados: {len(distritos_unicos)}")
    print(distritos_unicos)

    # Asegurar el formato numerico de las variables cuantitativas
    df_clima_diario['MAX'] = pd.to_numeric(df_clima_diario['MAX'], errors='coerce')
    df_clima_diario['MIN'] = pd.to_numeric(df_clima_diario['MIN'], errors='coerce')
    df_clima_diario['PRECIPITACION'] = pd.to_numeric(df_clima_diario['PRECIPITACION'], errors='coerce')

    # Filtro de anomalias fisicas en precipitacion
    df_clima_diario.loc[df_clima_diario['PRECIPITACION'] < 0, 'PRECIPITACION'] = np.nan
    df_clima_diario.loc[df_clima_diario['PRECIPITACION'] > 400, 'PRECIPITACION'] = np.nan

    # Interpolacion de valores faltantes y relleno de precipitacion
    df_clima_diario['MAX'] = df_clima_diario.groupby('DISTRITO')['MAX'].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    df_clima_diario['MIN'] = df_clima_diario.groupby('DISTRITO')['MIN'].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    df_clima_diario['PRECIPITACION'] = df_clima_diario.groupby('DISTRITO')['PRECIPITACION'].transform(lambda x: x.fillna(0))
    
    print(f"Unificacion de datos completada. Registros diarios totales: {len(df_clima_diario)}")
    
    # Agregacion temporal a frecuencia semanal (W-SUN)
    df_semanal_distritos = df_clima_diario.groupby([
        'DISTRITO',
        pd.Grouper(key='FECHA', freq='W-SUN')
    ]).agg(
        TEMP_MAX_SEM=('MAX', 'mean'),
        TEMP_MIN_SEM=('MIN', 'mean'),
        LLUVIA_TOTAL_SEM=('PRECIPITACION', 'sum')
    ).reset_index()
    
    # Calculo de las metricas regionales avanzadas
    df_regional_avanzado = df_semanal_distritos.groupby('FECHA').agg(
        Tmin_prom=('TEMP_MIN_SEM', 'mean'),
        Tmax_prom=('TEMP_MAX_SEM', 'mean'),
        P_total=('LLUVIA_TOTAL_SEM', 'mean'),
        P_max=('LLUVIA_TOTAL_SEM', 'max'),
        N_lluvia=('LLUVIA_TOTAL_SEM', lambda x: (x > 0.1).sum())
    ).reset_index()
    
    df_regional_avanzado = df_regional_avanzado.round(2)
    df_regional_avanzado.rename(columns={'FECHA': 'SEMANA_EPIDEMIOLOGICA'}, inplace=True)
    
    print("\nMatriz climatica regional avanzada:")
    display(df_regional_avanzado.head())
    
    # Exportacion de la matriz procesada
    os.makedirs("../data/processed", exist_ok=True)
    ruta_clima_procesado = "../data/processed/Clima_Regional_Avanzado.csv"
    df_regional_avanzado.to_csv(ruta_clima_procesado, index=False)
    print(f"Matriz climatica exportada a: {ruta_clima_procesado}")

Diagnostico - Numero de distritos procesados: 20
['AYABACA' 'BERNAL' 'CATACAOS' 'CHALACO' 'CHULUCANAS' 'COLAN' 'EL_ALTO'
 'FRIAS' 'HUANCABAMBA' 'HUARMACA' 'LANCONES' 'LAS_LOMAS' 'MARCAVELICA'
 'MORROPON' 'SALALA' 'SALITRAL' 'SANTO_DOMINGO' 'SAPILLICA' 'SECHURA'
 'SONDORILLO']
Unificacion de datos completada. Registros diarios totales: 15884

Matriz climatica regional avanzada:


,SEMANA_EPIDEMIOLOGICA,Tmin_prom,Tmax_prom,P_total,P_max,N_lluvia
0,2023-01-01,17.16,27.70,0.00,0.0,0
1,2023-01-08,15.55,28.27,3.50,13.8,9
2,2023-01-15,17.01,27.98,15.75,69.0,14
3,2023-01-22,16.83,28.24,17.57,68.5,12
4,2023-01-29,17.78,29.15,3.31,17.2,8


Matriz climatica exportada a: ../data/processed/Clima_Regional_Avanzado.csv


In [5]:
# FUSION EPIDEMIOLOGICA E INGENIERIA DE CARACTERISTICAS (Lags)
print("Iniciando fusion epidemiologica y generacion de rezagos temporales...\n")

if os.path.exists(ruta_dengue_raw) and 'ruta_clima_procesado' in locals():
    # Carga y limpieza de datos epidemiologicos
    df_dengue = pd.read_csv(ruta_dengue_raw, sep=r'[,;\t]+', engine='python', encoding='utf-8')
    df_dengue['FECHA_NOTIFICACION'] = pd.to_datetime(df_dengue['FECHA_NOTIFICACION'].astype(str), format='%Y%m%d', errors='coerce')
    df_dengue = df_dengue.dropna(subset=['FECHA_NOTIFICACION'])
    
    # Agregacion de casos por semana epidemiologica
    df_dengue_semanal = df_dengue.groupby(pd.Grouper(key='FECHA_NOTIFICACION', freq='W-SUN')).size().reset_index(name='CASOS_DENGUE')
    df_dengue_semanal.rename(columns={'FECHA_NOTIFICACION': 'SEMANA_EPIDEMIOLOGICA'}, inplace=True)
    
    # Fusion de datos climaticos y epidemiologicos
    df_clima = pd.read_csv(ruta_clima_procesado)
    df_clima['SEMANA_EPIDEMIOLOGICA'] = pd.to_datetime(df_clima['SEMANA_EPIDEMIOLOGICA'])
    
    df_final = pd.merge(df_clima, df_dengue_semanal, on='SEMANA_EPIDEMIOLOGICA', how='left')
    df_final['CASOS_DENGUE'] = df_final['CASOS_DENGUE'].fillna(0).astype(int)
    
    df_final = df_final.sort_values('SEMANA_EPIDEMIOLOGICA').reset_index(drop=True)
    
    # Generacion de lags (t-1, t-2) para la variable casos
    df_final['CASOS_t_1'] = df_final['CASOS_DENGUE'].shift(1)
    df_final['CASOS_t_2'] = df_final['CASOS_DENGUE'].shift(2)
    
    # Generacion de lags (t-1, t-2) para variables climaticas
    vars_clima = ['Tmin_prom', 'Tmax_prom', 'P_total', 'P_max', 'N_lluvia']
    for var in vars_clima:
        df_final[f'{var}_t_1'] = df_final[var].shift(1)
        df_final[f'{var}_t_2'] = df_final[var].shift(2)
        
    # Definicion de la variable objetivo predictiva (t+1)
    df_final['CASOS_PROXIMA_SEMANA_t_1'] = df_final['CASOS_DENGUE'].shift(-1)
    
    df_master = df_final.dropna().reset_index(drop=True)
    
    columnas_ordenadas = [
        'SEMANA_EPIDEMIOLOGICA',
        'CASOS_DENGUE', 'CASOS_t_1', 'CASOS_t_2',
        'Tmin_prom', 'Tmin_prom_t_1', 'Tmin_prom_t_2',
        'Tmax_prom', 'Tmax_prom_t_1', 'Tmax_prom_t_2',
        'P_total', 'P_total_t_1', 'P_total_t_2',
        'P_max', 'P_max_t_1', 'P_max_t_2',
        'N_lluvia', 'N_lluvia_t_1', 'N_lluvia_t_2',
        'CASOS_PROXIMA_SEMANA_t_1'
    ]
    df_master = df_master[columnas_ordenadas]
    
    print("Fusion completada y generacion de variables temporales finalizada.")
    display(df_master.head())
    
    ruta_master = "../data/processed/Dataset_Master_Dengue_Clima.csv"
    df_master.to_csv(ruta_master, index=False)
else:
    print("Faltan archivos fuente para llevar a cabo la fusion de datos.")

Iniciando fusion epidemiologica y generacion de rezagos temporales...

Fusion completada y generacion de variables temporales finalizada.


,SEMANA_EPIDEMIOLOGICA,CASOS_DENGUE,CASOS_t_1,CASOS_t_2,Tmin_prom,Tmin_prom_t_1,Tmin_prom_t_2,Tmax_prom,Tmax_prom_t_1,Tmax_prom_t_2,P_total,P_total_t_1,P_total_t_2,P_max,P_max_t_1,P_max_t_2,N_lluvia,N_lluvia_t_1,N_lluvia_t_2,CASOS_PROXIMA_SEMANA_t_1
0,2023-01-15,67,28.0,0.0,17.01,15.55,17.16,27.98,28.27,27.70,15.75,3.50,0.00,69.0,13.8,0.0,14,9.0,0.0,77.0
1,2023-01-22,77,67.0,28.0,16.83,17.01,15.55,28.24,27.98,28.27,17.57,15.75,3.50,68.5,69.0,13.8,12,14.0,9.0,140.0
2,2023-01-29,140,77.0,67.0,17.78,16.83,17.01,29.15,28.24,27.98,3.31,17.57,15.75,17.2,68.5,69.0,8,12.0,14.0,203.0
3,2023-02-05,203,140.0,77.0,18.29,17.78,16.83,28.62,29.15,28.24,7.81,3.31,17.57,38.6,17.2,68.5,14,8.0,12.0,208.0
4,2023-02-12,208,203.0,140.0,18.88,18.29,17.78,28.70,28.62,29.15,12.76,7.81,3.31,34.4,38.6,17.2,17,14.0,8.0,227.0


In [6]:
# ETIQUETADO DE RIESGO EPIDEMIOLOGICO (Evitando Data Leakage)
if 'df_master' in locals():
    # Eliminacion del periodo inactivo inicial
    primer_indice_valido = df_master[df_master['CASOS_DENGUE'] > 10].index[0]
    df_limpio = df_master.iloc[primer_indice_valido:].reset_index(drop=True)
    
    print(f"Se removieron los primeros {primer_indice_valido} registros por inactividad epidemiologica.")
    print(f"Fecha de inicio efectiva del dataset: {df_limpio['SEMANA_EPIDEMIOLOGICA'].dt.date.iloc[0]}")
    
    # Calculo de percentiles sobre el conjunto de entrenamiento (80%)
    corte_entrenamiento = int(len(df_limpio) * 0.80)
    df_train = df_limpio.iloc[:corte_entrenamiento]
    
    limite_bajo = df_train['CASOS_PROXIMA_SEMANA_t_1'].quantile(0.33)
    limite_alto = df_train['CASOS_PROXIMA_SEMANA_t_1'].quantile(0.66)
    
    print(f"\nParametros de clasificacion de riesgo calculados sobre el conjunto de entrenamiento:")
    print(f" Riesgo Bajo:   0 a {int(limite_bajo)} casos")
    print(f" Riesgo Medio:  {int(limite_bajo) + 1} a {int(limite_alto)} casos")
    print(f" Riesgo Alto:   > {int(limite_alto)} casos")
    
    def clasificar_riesgo(casos):
        if casos <= limite_bajo: return 'Bajo'
        elif casos <= limite_alto: return 'Medio'
        else: return 'Alto'
        
    df_limpio['NIVEL_INCIDENCIA'] = df_limpio['CASOS_PROXIMA_SEMANA_t_1'].apply(clasificar_riesgo)
    
    print("\nMuestra del conjunto de datos etiquetado final:")
    display(df_limpio[['SEMANA_EPIDEMIOLOGICA', 'CASOS_PROXIMA_SEMANA_t_1', 'NIVEL_INCIDENCIA']].head())
    
    ruta_export = "../data/processed/Dataset_Master_Etiquetado.csv"
    df_limpio.to_csv(ruta_export, index=False)
    print(f"\nExportacion del dataset final completada en: {ruta_export}")

Se removieron los primeros 0 registros por inactividad epidemiologica.
Fecha de inicio efectiva del dataset: 2023-01-15

Parametros de clasificacion de riesgo calculados sobre el conjunto de entrenamiento:
 Riesgo Bajo:   0 a 83 casos
 Riesgo Medio:  84 a 324 casos
 Riesgo Alto:   > 324 casos

Muestra del conjunto de datos etiquetado final:


,SEMANA_EPIDEMIOLOGICA,CASOS_PROXIMA_SEMANA_t_1,NIVEL_INCIDENCIA
0,2023-01-15,77.0,Bajo
1,2023-01-22,140.0,Medio
2,2023-01-29,203.0,Medio
3,2023-02-05,208.0,Medio
4,2023-02-12,227.0,Medio



Exportacion del dataset final completada en: ../data/processed/Dataset_Master_Etiquetado.csv
